In [ ]:
# ==========================================================
# Section 1: Project Configuration & Dataset Loading
# ==========================================================
#
# Purpose
# -------
# Initialise the project environment and load the frozen
# multi-task datasets for misinformation-tuned training.
#
# This section:
#   • Configures project directories
#   • Sets reproducibility parameters
#   • Configures logging
#   • Loads the frozen multi-task datasets
#   • Loads class weights
#   • Loads dataset metadata
#
# Inputs
# ------
# data/
# └── processed/
#     └── multitask/
#         multitask_train.csv
#         multitask_validation.csv
#         multitask_test.csv
#         multitask_class_weights.json
#         multitask_metadata.json
#
# Outputs
# -------
# In-memory datasets ready for tokenisation.
#
# ==========================================================

from pathlib import Path
import json
import logging
import random

import numpy as np
import pandas as pd
import torch

# ==========================================================
# Reproducibility
# ==========================================================

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(RANDOM_STATE)

# ==========================================================
# Project Structure
# ==========================================================

PROJECT_ROOT = Path.cwd().resolve()

DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DIR = DATA_DIR / "processed"

MULTITASK_DIR = PROCESSED_DIR / "multitask"

MODELS_DIR = PROJECT_ROOT / "models"

TUNED_DIR = MODELS_DIR / "misinformation_tuned"

CHECKPOINT_DIR = TUNED_DIR / "checkpoints"

TOKENIZER_DIR = TUNED_DIR / "tokenizer"

FIGURES_DIR = TUNED_DIR / "figures"

REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [

    MODELS_DIR,

    TUNED_DIR,

    CHECKPOINT_DIR,

    TOKENIZER_DIR,

    FIGURES_DIR,

    REPORTS_DIR,

]:

    directory.mkdir(

        parents=True,

        exist_ok=True

    )

# ==========================================================
# Device Configuration
# ==========================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

logger = logging.getLogger(__name__)

# ==========================================================
# Logging
# ==========================================================

logging.basicConfig(

    level=logging.INFO,

    format="%(levelname)s | %(message)s"

)

logger.info("=" * 70)
logger.info("MISINFORMATION-TUNED MULTI-TASK TRAINING")
logger.info("=" * 70)

logger.info(f"Device: {DEVICE}")

# ==========================================================
# Expected Input Files
# ==========================================================

required_files = {

    "Train":

        MULTITASK_DIR /

        "multitask_train.csv",

    "Validation":

        MULTITASK_DIR /

        "multitask_validation.csv",

    "Test":

        MULTITASK_DIR /

        "multitask_test.csv",

    "Class Weights":

        MULTITASK_DIR /

        "multitask_class_weights.json",

    "Metadata":

        MULTITASK_DIR /

        "multitask_metadata.json"

}

missing = [

    name

    for name, path in required_files.items()

    if not path.exists()

]

if missing:

    raise FileNotFoundError(

        "The following required files are missing:\n\n"

        + "\n".join(missing)

        + "\n\nRun 03_build_multitask_dataset.ipynb first."

    )

logger.info("✓ All required datasets located.")

# ==========================================================
# Load Frozen Datasets
# ==========================================================

logger.info("=" * 70)
logger.info("LOADING MULTI-TASK DATASETS")
logger.info("=" * 70)

train_df = pd.read_csv(

    required_files["Train"]

)

validation_df = pd.read_csv(

    required_files["Validation"]

)

test_df = pd.read_csv(

    required_files["Test"]

)

logger.info(

    f"Training samples    : {len(train_df):,}"

)

logger.info(

    f"Validation samples  : {len(validation_df):,}"

)

logger.info(

    f"Test samples        : {len(test_df):,}"

)

# ==========================================================
# Load Class Weights
# ==========================================================

logger.info("Loading class weights...")

with open(

    required_files["Class Weights"],

    "r",

    encoding="utf-8"

) as fp:

    multitask_class_weights = json.load(fp)

logger.info("✓ Class weights loaded.")

# ==========================================================
# Load Dataset Metadata
# ==========================================================

logger.info("Loading dataset metadata...")

with open(

    required_files["Metadata"],

    "r",

    encoding="utf-8"

) as fp:

    multitask_metadata = json.load(fp)

logger.info("✓ Metadata loaded.")

# ==========================================================
# Hyperparameters
# ==========================================================

MODEL_NAME = "xlm-roberta-base"

MAX_LENGTH = 256

BATCH_SIZE = 16

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

NUM_EPOCHS = 10

EARLY_STOPPING_PATIENCE = 3

GRADIENT_CLIP = 1.0

# ==========================================================
# Misinformation-Tuned Loss Weights
# ==========================================================

MISINFO_ALPHA = 2.0

HATE_BETA = 1.0

logger.info("=" * 70)
logger.info("TRAINING CONFIGURATION")
logger.info("=" * 70)

configuration = pd.DataFrame({

    "Parameter": [

        "Model",

        "Device",

        "Batch Size",

        "Maximum Length",

        "Learning Rate",

        "Weight Decay",

        "Epochs",

        "Early Stopping",

        "Misinformation α",

        "Hate Speech β"

    ],

    "Value": [

        MODEL_NAME,

        str(DEVICE),

        BATCH_SIZE,

        MAX_LENGTH,

        LEARNING_RATE,

        WEIGHT_DECAY,

        NUM_EPOCHS,

        EARLY_STOPPING_PATIENCE,

        MISINFO_ALPHA,

        HATE_BETA

    ]

})

print("\nTraining Configuration\n")

print(configuration)

# ==========================================================
# Dataset Overview
# ==========================================================

logger.info("=" * 70)
logger.info("DATASET OVERVIEW")
logger.info("=" * 70)

overview = pd.DataFrame({

    "Dataset": [

        "Training",

        "Validation",

        "Test"

    ],

    "Samples": [

        len(train_df),

        len(validation_df),

        len(test_df)

    ]

})

print(overview)

logger.info("=" * 70)
logger.info("SECTION 1 COMPLETE")
logger.info("=" * 70)

logger.info(

    "Datasets loaded successfully."

)

logger.info(

    "Ready for tokenisation."

)

In [ ]:
# ==========================================================
# Section 2: Tokenisation & Dataset Construction
# ==========================================================
#
# Purpose
# -------
# Tokenise the multi-task datasets and construct PyTorch
# datasets and dataloaders for misinformation-tuned training.
#
# This section:
#   • Loads the XLM-RoBERTa tokenizer
#   • Defines the custom Dataset class
#   • Tokenises all dataset splits
#   • Creates DataLoaders
#   • Verifies batch integrity
#
# Outputs
# -------
# tokenizer
# train_dataset
# validation_dataset
# test_dataset
# train_loader
# validation_loader
# test_loader
#
# ==========================================================

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

logger.info("=" * 70)
logger.info("TOKENISATION & DATASET CONSTRUCTION")
logger.info("=" * 70)

# ==========================================================
# Load Tokenizer
# ==========================================================

logger.info("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

logger.info("✓ Tokenizer loaded.")

# ==========================================================
# Multi-task Dataset
# ==========================================================

class MultiTaskDataset(Dataset):
    """
    Dataset for multi-task learning.

    Expected columns
    ----------------

    text

    task_id

    task_label
    """

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length
    ):

        self.dataframe = dataframe.reset_index(
            drop=True
        )

        self.tokenizer = tokenizer

        self.max_length = max_length

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        encoding = self.tokenizer(

            row["text"],

            truncation=True,

            padding="max_length",

            max_length=self.max_length,

            return_attention_mask=True,

            return_tensors="pt"

        )

        return {

            "input_ids":

                encoding["input_ids"].squeeze(0),

            "attention_mask":

                encoding["attention_mask"].squeeze(0),

            "task_id":

                torch.tensor(

                    row["task_id"],

                    dtype=torch.long

                ),

            "task_label":

                torch.tensor(

                    row["task_label"],

                    dtype=torch.long

                )

        }

# ==========================================================
# Construct Datasets
# ==========================================================

logger.info("Creating datasets...")

train_dataset = MultiTaskDataset(

    train_df,

    tokenizer,

    MAX_LENGTH

)

validation_dataset = MultiTaskDataset(

    validation_df,

    tokenizer,

    MAX_LENGTH

)

test_dataset = MultiTaskDataset(

    test_df,

    tokenizer,

    MAX_LENGTH

)

logger.info("✓ Dataset objects created.")

# ==========================================================
# Create DataLoaders
# ==========================================================

logger.info("Creating dataloaders...")

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    drop_last=False

)

validation_loader = DataLoader(

    validation_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    drop_last=False

)

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    drop_last=False

)

logger.info("✓ DataLoaders created.")

# ==========================================================
# Verify Dataset Sizes
# ==========================================================

summary = pd.DataFrame({

    "Dataset": [

        "Training",

        "Validation",

        "Test"

    ],

    "Samples": [

        len(train_dataset),

        len(validation_dataset),

        len(test_dataset)

    ],

    "Batches": [

        len(train_loader),

        len(validation_loader),

        len(test_loader)

    ]

})

print("\nDataset Summary\n")

print(summary)

# ==========================================================
# Batch Verification
# ==========================================================

logger.info("=" * 70)
logger.info("VERIFYING DATA PIPELINE")
logger.info("=" * 70)

sample_batch = next(iter(train_loader))

print("\nBatch Shapes\n")

print(

    "Input IDs      :",

    tuple(sample_batch["input_ids"].shape)

)

print(

    "Attention Mask :",

    tuple(sample_batch["attention_mask"].shape)

)

print(

    "Task IDs       :",

    tuple(sample_batch["task_id"].shape)

)

print(

    "Task Labels    :",

    tuple(sample_batch["task_label"].shape)

)

logger.info("✓ Batch verification successful.")

# ==========================================================
# Tokenisation Statistics
# ==========================================================

logger.info("=" * 70)
logger.info("TOKENISATION SUMMARY")
logger.info("=" * 70)

logger.info(f"Tokenizer      : {MODEL_NAME}")

logger.info(f"Maximum Length : {MAX_LENGTH}")

logger.info(f"Batch Size     : {BATCH_SIZE}")

logger.info(f"Training Batches    : {len(train_loader)}")

logger.info(f"Validation Batches : {len(validation_loader)}")

logger.info(f"Test Batches       : {len(test_loader)}")

logger.info("=" * 70)
logger.info("SECTION 2 COMPLETE")
logger.info("=" * 70)

logger.info(

    "Tokenisation completed successfully."

)

logger.info(

    "Ready for multi-task model definition."

)

In [ ]:
# ==========================================================
# Section 3: Multi-task Model Definition
# ==========================================================
#
# Purpose
# -------
# Define the shared XLM-RoBERTa encoder and the task-specific
# classification heads used for misinformation-tuned
# multi-task learning.
#
# This section:
#   • Loads the pretrained XLM-RoBERTa encoder
#   • Defines two task-specific classification heads
#   • Creates the complete multi-task model
#   • Moves the model to the selected device
#   • Reports model statistics
#
# Outputs
# -------
# multitask_model
#
# ==========================================================

import torch
import torch.nn as nn

from transformers import AutoModel

logger.info("=" * 70)
logger.info("MULTI-TASK MODEL DEFINITION")
logger.info("=" * 70)

# ==========================================================
# Multi-task XLM-RoBERTa Model
# ==========================================================

class MultiTaskXLMRoberta(nn.Module):

    """
    Shared XLM-RoBERTa encoder with two task-specific
    classification heads.

    Tasks
    -----

    Head 1
        Binary misinformation detection

    Head 2
        Three-class hate speech classification
    """

    def __init__(

        self,

        model_name

    ):

        super().__init__()

        # --------------------------------------------------
        # Shared Encoder
        # --------------------------------------------------

        self.encoder = AutoModel.from_pretrained(

            model_name

        )

        hidden_size = (

            self.encoder.config.hidden_size

        )

        dropout = (

            self.encoder.config.hidden_dropout_prob

        )

        # --------------------------------------------------
        # Shared Dropout
        # --------------------------------------------------

        self.dropout = nn.Dropout(

            dropout

        )

        # --------------------------------------------------
        # Misinformation Head
        # --------------------------------------------------

        self.misinformation_classifier = nn.Linear(

            hidden_size,

            2

        )

        # --------------------------------------------------
        # Hate Speech Head
        # --------------------------------------------------

        self.hate_classifier = nn.Linear(

            hidden_size,

            3

        )

    # ------------------------------------------------------
    # Forward Pass
    # ------------------------------------------------------

    def forward(

        self,

        input_ids,

        attention_mask

    ):

        outputs = self.encoder(

            input_ids=input_ids,

            attention_mask=attention_mask

        )

        pooled_output = outputs.last_hidden_state[:, 0]

        pooled_output = self.dropout(

            pooled_output

        )

        misinformation_logits = (

            self.misinformation_classifier(

                pooled_output

            )

        )

        hate_logits = (

            self.hate_classifier(

                pooled_output

            )

        )

        return {

            "misinformation":

                misinformation_logits,

            "hate":

                hate_logits

        }

# ==========================================================
# Instantiate Model
# ==========================================================

logger.info("Loading pretrained XLM-RoBERTa...")

multitask_model = MultiTaskXLMRoberta(

    MODEL_NAME

)

multitask_model.to(

    DEVICE

)

logger.info("✓ Model initialised.")

# ==========================================================
# Model Statistics
# ==========================================================

total_parameters = sum(

    parameter.numel()

    for parameter

    in multitask_model.parameters()

)

trainable_parameters = sum(

    parameter.numel()

    for parameter

    in multitask_model.parameters()

    if parameter.requires_grad

)

encoder_parameters = sum(

    parameter.numel()

    for parameter

    in multitask_model.encoder.parameters()

)

misinformation_head_parameters = sum(

    parameter.numel()

    for parameter

    in multitask_model

    .misinformation_classifier

    .parameters()

)

hate_head_parameters = sum(

    parameter.numel()

    for parameter

    in multitask_model

    .hate_classifier

    .parameters()

)

statistics = pd.DataFrame({

    "Component": [

        "Encoder",

        "Misinformation Head",

        "Hate Speech Head",

        "Total Trainable",

        "Total Parameters"

    ],

    "Parameters": [

        f"{encoder_parameters:,}",

        f"{misinformation_head_parameters:,}",

        f"{hate_head_parameters:,}",

        f"{trainable_parameters:,}",

        f"{total_parameters:,}"

    ]

})

print("\nModel Summary\n")

print(statistics)

# ==========================================================
# Architecture Validation
# ==========================================================

logger.info("=" * 70)
logger.info("VERIFYING MODEL ARCHITECTURE")
logger.info("=" * 70)

dummy_batch = next(

    iter(train_loader)

)

with torch.no_grad():

    outputs = multitask_model(

        input_ids=dummy_batch["input_ids"].to(DEVICE),

        attention_mask=dummy_batch["attention_mask"].to(DEVICE)

    )

print("\nOutput Shapes\n")

print(

    "Misinformation:",

    tuple(

        outputs["misinformation"].shape

    )

)

print(

    "Hate Speech:",

    tuple(

        outputs["hate"].shape

    )

)

assert (

    outputs["misinformation"].shape[1] == 2

)

assert (

    outputs["hate"].shape[1] == 3

)

logger.info("✓ Output dimensions verified.")

logger.info("=" * 70)
logger.info("SECTION 3 COMPLETE")
logger.info("=" * 70)

logger.info(

    "Multi-task model created successfully."

)

logger.info(

    "Ready for loss function configuration."

)

In [ ]:
# ==========================================================
# Section 4: Loss Functions & Optimiser
# ==========================================================
#
# Purpose
# -------
# Configure the loss functions, optimiser and learning-rate
# scheduler for the misinformation-tuned multi-task model.
#
# This section:
#   • Loads task-specific class weights
#   • Creates weighted loss functions
#   • Applies misinformation-focused loss weighting
#   • Configures the AdamW optimiser
#   • Configures the learning-rate scheduler
#
# Outputs
# -------
# misinformation_criterion
# hate_criterion
# optimiser
# scheduler
#
# ==========================================================

import torch
import torch.nn as nn

from transformers import get_linear_schedule_with_warmup

logger.info("=" * 70)
logger.info("LOSS FUNCTIONS & OPTIMISER")
logger.info("=" * 70)

# ==========================================================
# Load Task-specific Class Weights
# ==========================================================

logger.info("Loading class weights...")

misinformation_weights = torch.tensor(

    [

        multitask_class_weights["misinformation"]["0"],

        multitask_class_weights["misinformation"]["1"]

    ],

    dtype=torch.float

).to(DEVICE)

hate_weights = torch.tensor(

    [

        multitask_class_weights["hate"]["0"],

        multitask_class_weights["hate"]["1"],

        multitask_class_weights["hate"]["2"]

    ],

    dtype=torch.float

).to(DEVICE)

logger.info("✓ Class weights loaded.")

# ==========================================================
# Loss Functions
# ==========================================================

misinformation_criterion = nn.CrossEntropyLoss(

    weight=misinformation_weights

)

hate_criterion = nn.CrossEntropyLoss(

    weight=hate_weights

)

logger.info("✓ Weighted CrossEntropy losses created.")

# ==========================================================
# Multi-task Loss Configuration
# ==========================================================
#
# Total Loss
#
#     α × misinformation_loss
#   + β × hate_loss
#
# ==========================================================

logger.info("=" * 70)
logger.info("MISINFORMATION-TUNED LOSS CONFIGURATION")
logger.info("=" * 70)

logger.info(

    f"Misinformation α = {MISINFO_ALPHA:.2f}"

)

logger.info(

    f"Hate Speech β    = {HATE_BETA:.2f}"

)

def compute_multitask_loss(

    misinformation_logits,

    misinformation_labels,

    hate_logits,

    hate_labels

):

    misinformation_loss = misinformation_criterion(

        misinformation_logits,

        misinformation_labels

    )

    hate_loss = hate_criterion(

        hate_logits,

        hate_labels

    )

    total_loss = (

        MISINFO_ALPHA * misinformation_loss

        +

        HATE_BETA * hate_loss

    )

    return (

        total_loss,

        misinformation_loss,

        hate_loss

    )

logger.info(

    "✓ Weighted multi-task loss configured."

)

# ==========================================================
# Optimiser
# ==========================================================

logger.info("Creating optimiser...")

optimizer = torch.optim.AdamW(

    multitask_model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY

)

logger.info("✓ AdamW optimiser created.")

# ==========================================================
# Learning-rate Scheduler
# ==========================================================

total_training_steps = (

    len(train_loader)

    * NUM_EPOCHS

)

warmup_steps = int(

    0.10 * total_training_steps

)

scheduler = get_linear_schedule_with_warmup(

    optimizer,

    num_warmup_steps=warmup_steps,

    num_training_steps=total_training_steps

)

logger.info("✓ Linear scheduler created.")

# ==========================================================
# Training Configuration
# ==========================================================

training_config = {

    "experiment":

        "misinformation_tuned",

    "model_name":

        MODEL_NAME,

    "random_state":

        RANDOM_STATE,

    "batch_size":

        BATCH_SIZE,

    "max_length":

        MAX_LENGTH,

    "learning_rate":

        LEARNING_RATE,

    "weight_decay":

        WEIGHT_DECAY,

    "epochs":

        NUM_EPOCHS,

    "warmup_steps":

        warmup_steps,

    "gradient_clip":

        GRADIENT_CLIP,

    "misinformation_alpha":

        MISINFO_ALPHA,

    "hate_beta":

        HATE_BETA,

    "optimizer":

        "AdamW",

    "scheduler":

        "Linear Warmup",

    "loss_function":

        "Weighted CrossEntropy"

}

print("\nTraining Configuration\n")

configuration = pd.DataFrame({

    "Parameter": [

        "Experiment",

        "Learning Rate",

        "Weight Decay",

        "Epochs",

        "Warmup Steps",

        "Misinformation α",

        "Hate Speech β"

    ],

    "Value": [

        training_config["experiment"],

        training_config["learning_rate"],

        training_config["weight_decay"],

        training_config["epochs"],

        training_config["warmup_steps"],

        training_config["misinformation_alpha"],

        training_config["hate_beta"]

    ]

})

print(configuration)

logger.info("=" * 70)
logger.info("SECTION 4 COMPLETE")
logger.info("=" * 70)

logger.info(

    "Loss functions and optimiser configured."

)

logger.info(

    "Ready for misinformation-tuned training."

)

In [ ]:
# ==========================================================
# Section 5: Training Initialisation
# ==========================================================
#
# Purpose
# -------
# Initialise the training process by preparing training
# history, checkpoint tracking, early stopping variables
# and timing utilities.
#
# ==========================================================

import copy
import time

from tqdm.auto import tqdm

logger.info("=" * 70)
logger.info("INITIALISING TRAINING")
logger.info("=" * 70)

# ==========================================================
# Training History
# ==========================================================

training_history = []

# ==========================================================
# Best Model Tracking
# ==========================================================

best_validation_loss = float("inf")

best_epoch = 0

best_model_state = None

# ==========================================================
# Early Stopping
# ==========================================================

epochs_without_improvement = 0

# ==========================================================
# Training Timer
# ==========================================================

training_start_time = time.time()

logger.info("Training history initialised.")

logger.info("Best checkpoint tracking initialised.")

logger.info("Early stopping configured.")

logger.info(

    f"Patience: {EARLY_STOPPING_PATIENCE} epochs"

)

# ==========================================================
# Epoch Loop
# ==========================================================

logger.info("=" * 70)
logger.info("BEGIN TRAINING")
logger.info("=" * 70)

for epoch in range(NUM_EPOCHS):

    logger.info(

        f"Epoch {epoch + 1}/{NUM_EPOCHS}"

    )

    logger.info("-" * 70)

    # ======================================================
    # Training Mode
    # ======================================================

    multitask_model.train()

    # ======================================================
    # Running Statistics
    # ======================================================

    running_total_loss = 0.0

    running_misinformation_loss = 0.0

    running_hate_loss = 0.0

    running_samples = 0

    # ======================================================
    # Progress Bar
    # ======================================================

    progress_bar = tqdm(

        train_loader,

        desc=f"Epoch {epoch + 1}",

        leave=False

    )

    # ======================================================
    # Batch Loop
    # ======================================================

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(

            DEVICE

        )

        attention_mask = batch["attention_mask"].to(

            DEVICE

        )

        task_ids = batch["task_id"].to(

            DEVICE

        )

        task_labels = batch["task_label"].to(

            DEVICE

        )

        optimizer.zero_grad()

                # ==================================================
        # Forward Pass
        # ==================================================

        outputs = multitask_model(

            input_ids=input_ids,

            attention_mask=attention_mask

        )

        # ==================================================
        # Separate Samples by Task
        # ==================================================

        misinfo_mask = (

            task_ids == 0

        )

        hate_mask = (

            task_ids == 1

        )

        total_loss = torch.tensor(

            0.0,

            device=DEVICE

        )

        misinfo_loss = torch.tensor(

            0.0,

            device=DEVICE

        )

        hate_loss = torch.tensor(

            0.0,

            device=DEVICE

        )

        # ==================================================
        # Misinformation Task
        # ==================================================

        if misinfo_mask.any():

            misinfo_logits = outputs[

                "misinformation"

            ][misinfo_mask]

            misinfo_labels = task_labels[

                misinfo_mask

            ]

            misinfo_loss = misinformation_criterion(

                misinfo_logits,

                misinfo_labels

            )

            total_loss += (

                MISINFO_ALPHA

                * misinfo_loss

            )

        # ==================================================
        # Hate Speech Task
        # ==================================================

        if hate_mask.any():

            hate_logits = outputs[

                "hate"

            ][hate_mask]

            hate_labels = task_labels[

                hate_mask

            ]

            hate_loss = hate_criterion(

                hate_logits,

                hate_labels

            )

            total_loss += (

                HATE_BETA

                * hate_loss

            )

        # ==================================================
        # Backpropagation
        # ==================================================

        total_loss.backward()

        # ==================================================
        # Gradient Clipping
        # ==================================================

        torch.nn.utils.clip_grad_norm_(

            multitask_model.parameters(),

            GRADIENT_CLIP

        )

        # ==================================================
        # Optimiser Step
        # ==================================================

        optimizer.step()

        scheduler.step()

        # ==================================================
        # Running Statistics
        # ==================================================

        batch_size = input_ids.size(0)

        running_samples += batch_size

        running_total_loss += (

            total_loss.item()

            * batch_size

        )

        running_misinformation_loss += (

            misinfo_loss.item()

            * batch_size

        )

        running_hate_loss += (

            hate_loss.item()

            * batch_size

        )

        # ==================================================
        # Progress Bar
        # ==================================================

        progress_bar.set_postfix({

            "Loss":

                f"{total_loss.item():.4f}",

            "Misinfo":

                f"{misinfo_loss.item():.4f}",

            "Hate":

                f"{hate_loss.item():.4f}"

        })

    # ======================================================
    # Epoch Training Statistics
    # ======================================================

    train_total_loss = (

        running_total_loss

        / running_samples

    )

    train_misinformation_loss = (

        running_misinformation_loss

        / running_samples

    )

    train_hate_loss = (

        running_hate_loss

        / running_samples

    )

    logger.info(

        f"Training Loss: "

        f"{train_total_loss:.4f}"

    )

    logger.info(

        f"Misinformation Loss: "

        f"{train_misinformation_loss:.4f}"

    )

    logger.info(

        f"Hate Speech Loss: "

        f"{train_hate_loss:.4f}"

    )

    # Validation loop begins in Part 5C.
    # ======================================================
    # Validation Phase
    # ======================================================

    multitask_model.eval()

    validation_total_loss = 0.0

    validation_misinformation_loss = 0.0

    validation_hate_loss = 0.0

    validation_samples = 0

    with torch.no_grad():

        for batch in tqdm(

            validation_loader,

            desc="Validation",

            leave=False

        ):

            input_ids = batch["input_ids"].to(DEVICE)

            attention_mask = batch["attention_mask"].to(DEVICE)

            task_ids = batch["task_id"].to(DEVICE)

            task_labels = batch["task_label"].to(DEVICE)

            outputs = multitask_model(

                input_ids=input_ids,

                attention_mask=attention_mask

            )

            misinfo_mask = task_ids == 0

            hate_mask = task_ids == 1

            total_loss = torch.tensor(

                0.0,

                device=DEVICE

            )

            misinfo_loss = torch.tensor(

                0.0,

                device=DEVICE

            )

            hate_loss = torch.tensor(

                0.0,

                device=DEVICE

            )

            # ==============================================
            # Misinformation Validation Loss
            # ==============================================

            if misinfo_mask.any():

                misinfo_logits = outputs[

                    "misinformation"

                ][misinfo_mask]

                misinfo_labels = task_labels[

                    misinfo_mask

                ]

                misinfo_loss = misinformation_criterion(

                    misinfo_logits,

                    misinfo_labels

                )

                total_loss += (

                    MISINFO_ALPHA

                    * misinfo_loss

                )

            # ==============================================
            # Hate Validation Loss
            # ==============================================

            if hate_mask.any():

                hate_logits = outputs[

                    "hate"

                ][hate_mask]

                hate_labels = task_labels[

                    hate_mask

                ]

                hate_loss = hate_criterion(

                    hate_logits,

                    hate_labels

                )

                total_loss += (

                    HATE_BETA

                    * hate_loss

                )

            batch_size = input_ids.size(0)

            validation_samples += batch_size

            validation_total_loss += (

                total_loss.item()

                * batch_size

            )

            validation_misinformation_loss += (

                misinfo_loss.item()

                * batch_size

            )

            validation_hate_loss += (

                hate_loss.item()

                * batch_size

            )

    # ======================================================
    # Epoch Validation Statistics
    # ======================================================

    validation_total_loss /= validation_samples

    validation_misinformation_loss /= validation_samples

    validation_hate_loss /= validation_samples

    logger.info(

        f"Validation Loss: "

        f"{validation_total_loss:.4f}"

    )

    logger.info(

        f"Misinformation Validation Loss: "

        f"{validation_misinformation_loss:.4f}"

    )

    logger.info(

        f"Hate Speech Validation Loss: "

        f"{validation_hate_loss:.4f}"

    )

    # ======================================================
    # Save Epoch History
    # ======================================================

    training_history.append({

        "epoch":

            epoch + 1,

        "train_total_loss":

            train_total_loss,

        "train_misinformation_loss":

            train_misinformation_loss,

        "train_hate_loss":

            train_hate_loss,

        "validation_total_loss":

            validation_total_loss,

        "validation_misinformation_loss":

            validation_misinformation_loss,

        "validation_hate_loss":

            validation_hate_loss

    })

    # ======================================================
    # Best Model Checkpoint
    # ======================================================

    if validation_total_loss < best_validation_loss:

        best_validation_loss = validation_total_loss

        best_epoch = epoch + 1

        epochs_without_improvement = 0

        best_model_state = copy.deepcopy(

            multitask_model.state_dict()

        )

        torch.save(

            best_model_state,

            CHECKPOINT_DIR /

            "best_model.pt"

        )

        logger.info(

            "✓ Best model checkpoint updated."

        )

    else:

        epochs_without_improvement += 1

        logger.info(

            f"No improvement "

            f"({epochs_without_improvement}/"

            f"{EARLY_STOPPING_PATIENCE})"

        )

    # ======================================================
    # Early Stopping
    # ======================================================

    if (

        epochs_without_improvement

        >=

        EARLY_STOPPING_PATIENCE

    ):

        logger.info(

            "=" * 70

        )

        logger.info(

            "EARLY STOPPING TRIGGERED"

        )

        logger.info(

            "=" * 70

        )

        break

# ==========================================================
# Save Training History
# ==========================================================

training_history = pd.DataFrame(

    training_history

)

training_history.to_csv(

    TUNED_DIR /

    "training_history.csv",

    index=False

)

logger.info(

    "✓ Training history saved."

)

# Part 5D continues from here.
# ==========================================================
# Part 5D: Restore Best Model & Final Summary
# ==========================================================
#
# Purpose
# -------
# Restore the best-performing model checkpoint and produce a
# final summary of the misinformation-tuned training process.
#
# Outputs
# -------
# In-memory best model ready for final evaluation.
#
# ==========================================================

logger.info("=" * 70)
logger.info("RESTORING BEST MODEL")
logger.info("=" * 70)

# ==========================================================
# Restore Best Checkpoint
# ==========================================================

best_checkpoint = CHECKPOINT_DIR / "best_model.pt"

if not best_checkpoint.exists():

    raise FileNotFoundError(

        "Best checkpoint was not found.\n"

        "Training may have terminated unexpectedly."

    )

multitask_model.load_state_dict(

    torch.load(

        best_checkpoint,

        map_location=DEVICE

    )

)

multitask_model.to(DEVICE)

multitask_model.eval()

logger.info("✓ Best checkpoint restored.")

# ==========================================================
# Training Duration
# ==========================================================

training_end_time = time.time()

training_time_seconds = (

    training_end_time

    - training_start_time

)

training_time_minutes = (

    training_time_seconds

    / 60

)

# ==========================================================
# Final Training Summary
# ==========================================================

logger.info("=" * 70)
logger.info("TRAINING SUMMARY")
logger.info("=" * 70)

summary = pd.DataFrame({

    "Metric": [

        "Experiment",

        "Best Epoch",

        "Best Validation Loss",

        "Epochs Completed",

        "Early Stopping Patience",

        "Training Time (minutes)",

        "Batch Size",

        "Learning Rate",

        "Misinformation α",

        "Hate Speech β"

    ],

    "Value": [

        EXPERIMENT_NAME,

        best_epoch,

        round(

            best_validation_loss,

            6

        ),

        len(training_history),

        EARLY_STOPPING_PATIENCE,

        round(

            training_time_minutes,

            2

        ),

        BATCH_SIZE,

        LEARNING_RATE,

        MISINFO_ALPHA,

        HATE_BETA

    ]

})

print("\nTraining Summary\n")

print(summary)

# ==========================================================
# Save Training Configuration
# ==========================================================

training_configuration = {

    "experiment": EXPERIMENT_NAME,

    "model": MODEL_NAME,

    "random_state": RANDOM_STATE,

    "device": str(DEVICE),

    "batch_size": BATCH_SIZE,

    "max_length": MAX_LENGTH,

    "learning_rate": LEARNING_RATE,

    "weight_decay": WEIGHT_DECAY,

    "epochs_requested": NUM_EPOCHS,

    "epochs_completed": len(training_history),

    "best_epoch": best_epoch,

    "best_validation_loss":

        float(best_validation_loss),

    "early_stopping_patience":

        EARLY_STOPPING_PATIENCE,

    "gradient_clip":

        GRADIENT_CLIP,

    "misinformation_alpha":

        MISINFO_ALPHA,

    "hate_beta":

        HATE_BETA,

    "training_time_seconds":

        round(training_time_seconds, 2)

}

with open(

    TUNED_DIR /

    "training_configuration.json",

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        training_configuration,

        fp,

        indent=4

    )

logger.info(

    "✓ Training configuration saved."

)

# ==========================================================
# Verify Generated Artefacts
# ==========================================================

logger.info("=" * 70)
logger.info("VERIFYING TRAINING ARTEFACTS")
logger.info("=" * 70)

generated_files = [

    "training_history.csv",

    "training_configuration.json",

    "checkpoints/best_model.pt"

]

for file in generated_files:

    path = TUNED_DIR / file

    if path.exists():

        logger.info(f"✓ {file}")

    else:

        logger.warning(f"✗ {file}")

# ==========================================================
# Completion Summary
# ==========================================================

logger.info("=" * 70)
logger.info("MISINFORMATION-TUNED TRAINING COMPLETE")
logger.info("=" * 70)

print("\nGenerated Artefacts\n")

print("Model Directory")

print(f"✓ {TUNED_DIR}")

print("\nCheckpoint")

print(f"✓ {best_checkpoint.name}")

print("\nFiles")

for file in sorted(

    TUNED_DIR.glob("*")

):

    if file.is_file():

        print(f"✓ {file.name}")

print("\nTraining completed successfully.")

logger.info(

    "Best model restored and ready "

    "for final evaluation."

)

logger.info(

    "Proceed to Section 6 "

    "(Final Evaluation)."

)



In [ ]:
# ==========================================================
# Section 6: Final Evaluation
# ==========================================================
#
# Purpose
# -------
# Evaluate the best-performing misinformation-tuned
# multi-task model on the held-out test dataset.
#
# This section:
#   • Generates predictions
#   • Computes evaluation metrics
#   • Produces confusion matrices
#   • Stores evaluation artefacts
#
# Outputs
# -------
# evaluation_results
# confusion_matrices
#
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

logger.info("=" * 70)
logger.info("FINAL MODEL EVALUATION")
logger.info("=" * 70)

multitask_model.eval()

misinfo_true = []
misinfo_pred = []

hate_true = []
hate_pred = []

with torch.no_grad():

    for batch in tqdm(
        test_loader,
        desc="Testing"
    ):

        input_ids = batch["input_ids"].to(DEVICE)

        attention_mask = batch["attention_mask"].to(DEVICE)

        task_ids = batch["task_id"].to(DEVICE)

        task_labels = batch["task_label"].to(DEVICE)

        outputs = multitask_model(

            input_ids=input_ids,

            attention_mask=attention_mask

        )

        # ----------------------------------------------
        # Misinformation Predictions
        # ----------------------------------------------

        misinfo_mask = task_ids == 0

        if misinfo_mask.any():

            predictions = torch.argmax(

                outputs["misinformation"][misinfo_mask],

                dim=1

            )

            misinfo_true.extend(

                task_labels[misinfo_mask]

                .cpu()

                .numpy()

            )

            misinfo_pred.extend(

                predictions

                .cpu()

                .numpy()

            )

        # ----------------------------------------------
        # Hate Speech Predictions
        # ----------------------------------------------

        hate_mask = task_ids == 1

        if hate_mask.any():

            predictions = torch.argmax(

                outputs["hate"][hate_mask],

                dim=1

            )

            hate_true.extend(

                task_labels[hate_mask]

                .cpu()

                .numpy()

            )

            hate_pred.extend(

                predictions

                .cpu()

                .numpy()

            )

logger.info("✓ Predictions generated successfully.")

# ==========================================================
# Evaluation Function
# ==========================================================

def evaluate_task(

    y_true,

    y_pred,

    task_name,

    target_names

):

    accuracy = accuracy_score(

        y_true,

        y_pred

    )

    precision, recall, f1, _ = (

        precision_recall_fscore_support(

            y_true,

            y_pred,

            average="macro",

            zero_division=0

        )

    )

    report = classification_report(

        y_true,

        y_pred,

        target_names=target_names,

        output_dict=True,

        zero_division=0

    )

    matrix = confusion_matrix(

        y_true,

        y_pred

    )

    logger.info("=" * 70)

    logger.info(task_name)

    logger.info("=" * 70)

    print(

        classification_report(

            y_true,

            y_pred,

            target_names=target_names,

            zero_division=0

        )

    )

    return {

        "accuracy": float(accuracy),

        "precision_macro": float(precision),

        "recall_macro": float(recall),

        "f1_macro": float(f1),

        "classification_report": report,

        "confusion_matrix": matrix.tolist()

    }

# ==========================================================
# Evaluate Misinformation Task
# ==========================================================

misinformation_results = evaluate_task(

    misinfo_true,

    misinfo_pred,

    "MISINFORMATION DETECTION",

    [

        "Not Misinformation",

        "Misinformation"

    ]

)

# ==========================================================
# Evaluate Hate Speech Task
# ==========================================================

hate_results = evaluate_task(

    hate_true,

    hate_pred,

    "HATE SPEECH DETECTION",

    [

        "Normal",

        "Abusive",

        "Hate"

    ]

)

# ==========================================================
# Overall Evaluation Summary
# ==========================================================

evaluation_results = {

    "misinformation":

        misinformation_results,

    "hate":

        hate_results

}

summary = pd.DataFrame({

    "Task": [

        "Misinformation",

        "Hate Speech"

    ],

    "Accuracy": [

        misinformation_results["accuracy"],

        hate_results["accuracy"]

    ],

    "Macro Precision": [

        misinformation_results["precision_macro"],

        hate_results["precision_macro"]

    ],

    "Macro Recall": [

        misinformation_results["recall_macro"],

        hate_results["recall_macro"]

    ],

    "Macro F1": [

        misinformation_results["f1_macro"],

        hate_results["f1_macro"]

    ]

})

logger.info("=" * 70)
logger.info("FINAL TEST RESULTS")
logger.info("=" * 70)

print(summary)

logger.info("=" * 70)
logger.info("SECTION 6 COMPLETE")
logger.info("=" * 70)

logger.info(

    "Misinformation-tuned model evaluation completed successfully."

)

In [ ]:
# ==========================================================
# Section 7: Export Artefacts
# ==========================================================
#
# Purpose
# -------
# Export all artefacts produced during the misinformation-
# tuned experiment.
#
# This section:
#   • Saves evaluation results
#   • Saves confusion matrices
#   • Saves the tokenizer
#   • Verifies exported artefacts
#
# Outputs
# -------
# models/
# └── misinformation_tuned/
#     ├── checkpoints/
#     │     best_model.pt
#     ├── tokenizer/
#     ├── training_history.csv
#     ├── training_configuration.json
#     ├── evaluation_results.json
#     └── confusion_matrices.json
#
# ==========================================================

import json

logger.info("=" * 70)
logger.info("EXPORTING EXPERIMENT ARTEFACTS")
logger.info("=" * 70)

# ==========================================================
# Save Evaluation Results
# ==========================================================

logger.info("Saving evaluation results...")

with open(

    TUNED_DIR /

    "evaluation_results.json",

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        evaluation_results,

        fp,

        indent=4

    )

logger.info("✓ Evaluation results saved.")

# ==========================================================
# Save Confusion Matrices
# ==========================================================

logger.info("Saving confusion matrices...")

confusion_matrices = {

    "misinformation":

        misinformation_results["confusion_matrix"],

    "hate":

        hate_results["confusion_matrix"]

}

with open(

    TUNED_DIR /

    "confusion_matrices.json",

    "w",

    encoding="utf-8"

) as fp:

    json.dump(

        confusion_matrices,

        fp,

        indent=4

    )

logger.info("✓ Confusion matrices saved.")

# ==========================================================
# Save Tokenizer
# ==========================================================

logger.info("Saving tokenizer...")

tokenizer.save_pretrained(

    TOKENIZER_DIR

)

logger.info("✓ Tokenizer saved.")

# ==========================================================
# Verify Exported Artefacts
# ==========================================================

logger.info("=" * 70)
logger.info("VERIFYING EXPORTED ARTEFACTS")
logger.info("=" * 70)

generated_files = [

    "training_history.csv",

    "training_configuration.json",

    "evaluation_results.json",

    "confusion_matrices.json"

]

for file in generated_files:

    path = TUNED_DIR / file

    if path.exists():

        logger.info(f"✓ {file}")

    else:

        logger.warning(f"✗ {file}")

checkpoint = (

    CHECKPOINT_DIR /

    "best_model.pt"

)

if checkpoint.exists():

    logger.info("✓ best_model.pt")

else:

    logger.warning("✗ best_model.pt")

if TOKENIZER_DIR.exists():

    logger.info("✓ tokenizer/")

else:

    logger.warning("✗ tokenizer/")

# ==========================================================
# Completion Summary
# ==========================================================

logger.info("=" * 70)
logger.info("MISINFORMATION-TUNED TRAINING COMPLETE")
logger.info("=" * 70)

print("\nGenerated Artefacts\n")

print("Model Directory")

print(f"✓ {TUNED_DIR}")

print("\nFiles")

for file in sorted(

    TUNED_DIR.glob("*")

):

    if file.is_file():

        print(f"✓ {file.name}")

print("\nCheckpoint")

print(

    f"✓ {CHECKPOINT_DIR / 'best_model.pt'}"

)

print("\nTokenizer")

print(f"✓ {TOKENIZER_DIR}")

print("\nExperiment completed successfully.")

logger.info(

    "All misinformation-tuned artefacts exported."

)

logger.info(

    "Notebook completed successfully."

)